# Day 10: Discord Lead - Chemprop + TabPFN (0.44 MAE)

**Discord comment:**
> "My activity submission is at 0.44 MAE range using chemprop+tabpfn ensembles and without using the single-dose and counter-screen data (yet) nor any PXR structural information."

**Key insights:**
1. They achieved **0.44 MAE** (vs our 0.52 MAE / 0.577 RAE)
2. Used **chemprop + TabPFN ensemble**
3. Did NOT use:
   - Single-dose data
   - Counter-screen data
   - PXR structural information
4. This suggests training on **~2800 molecules** that passed counter-screen

**Investigation plan:**
1. Filter to only molecules that passed counter-screen
2. Train Chemprop (message passing NN)
3. Train TabPFN (tabular prior-fitted networks)
4. Ensemble predictions
5. Compare with our baseline (LGBM + RDKit + Morgan FP)

**Target: 0.44-0.48 MAE**

In [ ]:
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from pathlib import Path

# RDKit
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.Scaffolds import MurckoScaffold

# ML
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
tqdm.pandas()
sns.set_style("whitegrid")

print("✅ Imports complete")

## 1. Load Data and Understand Counter-Screen

In [ ]:
# Load full training data
train_df = pd.read_csv("hf://datasets/openadmet/pxr-challenge-train-test/pxr-challenge_TRAIN.csv")
test_df = pd.read_csv("hf://datasets/openadmet/pxr-challenge-train-test/pxr-challenge_TEST_BLINDED.csv")

print(f"Full training set: {len(train_df)} molecules")
print(f"Test set: {len(test_df)} molecules")
print(f"\nColumns: {list(train_df.columns)}")

# Check counter-screen column
print(f"\nCounter-screen info:")
print(train_df['Counter-screen (1=Active, 0=Inactive)'].value_counts())

# Filter to only molecules that passed counter-screen (inactive in counter = active in PXR)
train_passed = train_df[train_df['Counter-screen (1=Active, 0=Inactive)'] == 0].copy()
print(f"\n✅ Molecules that passed counter-screen: {len(train_passed)}")
print(f"✅ Molecules filtered out: {len(train_df) - len(train_passed)}")

print(f"\nTarget distribution (passed counter-screen only):")
print(train_passed['pEC50'].describe())

## 2. Scaffold Groups for CV

In [ ]:
def get_scaffold(smiles):
    """Get Bemis-Murcko scaffold."""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return "invalid"
        scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
        return scaffold
    except:
        return "invalid"

def calculate_rae(y_true, y_pred):
    """Calculate Relative Absolute Error."""
    numerator = np.sum(np.abs(y_true - y_pred))
    denominator = np.sum(np.abs(y_true - np.mean(y_true)))
    return numerator / denominator if denominator != 0 else np.inf

print("Computing scaffolds...")
train_passed["scaffold"] = [get_scaffold(s) for s in tqdm(train_passed.SMILES)]
test_df["scaffold"] = [get_scaffold(s) for s in tqdm(test_df.SMILES)]

scaffold_to_group = {s: i for i, s in enumerate(train_passed["scaffold"].unique())}
train_passed["scaffold_group"] = train_passed["scaffold"].map(scaffold_to_group)

print(f"\nUnique scaffolds: {len(scaffold_to_group)}")

## 3. Install and Setup Chemprop

In [ ]:
# Check if chemprop is available
try:
    import chemprop
    print(f"✅ Chemprop version: {chemprop.__version__}")
    CHEMPROP_AVAILABLE = True
except ImportError:
    print("❌ Chemprop not installed. Installing...")
    !pip install chemprop -q
    import chemprop
    print(f"✅ Chemprop version: {chemprop.__version__}")
    CHEMPROP_AVAILABLE = True

## 4. Train Chemprop with 5-Fold CV

In [ ]:
from chemprop import data as cpdata, models as cpmodels, featurizers
import chemprop.nn as cpnn
import lightning as pl
import torch

y_train = train_passed['pEC50'].values
scaffold_groups = train_passed['scaffold_group'].values
group_kfold = GroupKFold(n_splits=5)

print("="*80)
print("CHEMPROP: 5-FOLD SCAFFOLD-GROUPED CROSS-VALIDATION")
print("="*80)

chemprop_oof_preds = np.zeros(len(train_passed))
chemprop_fold_results = []

for fold, (train_idx, val_idx) in enumerate(group_kfold.split(y_train, y_train, scaffold_groups), 1):
    print(f"\nFold {fold}/5...")
    
    train_smiles = train_passed['SMILES'].iloc[train_idx].tolist()
    val_smiles = train_passed['SMILES'].iloc[val_idx].tolist()
    train_targets = y_train[train_idx].reshape(-1, 1).tolist()
    val_targets = y_train[val_idx].reshape(-1, 1).tolist()
    
    featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer()
    
    train_data = cpdata.MoleculeDataset(
        [cpdata.MoleculeDatapoint.from_smi(smi, y)
         for smi, y in zip(train_smiles, train_targets)],
        featurizer=featurizer
    )
    val_data = cpdata.MoleculeDataset(
        [cpdata.MoleculeDatapoint.from_smi(smi, y)
         for smi, y in zip(val_smiles, val_targets)],
        featurizer=featurizer
    )
    
    train_loader = cpdata.build_dataloader(train_data, shuffle=True, batch_size=50)
    val_loader = cpdata.build_dataloader(val_data, shuffle=False, batch_size=50)
    
    mp = cpnn.BondMessagePassing()
    agg = cpnn.MeanAggregation()
    ffn = cpnn.RegressionFFN()
    model = cpmodels.MPNN(mp, agg, ffn)
    
    trainer = pl.Trainer(
        max_epochs=50,  # Increased from 30
        enable_progress_bar=False,
        enable_model_summary=False,
    )
    trainer.fit(model, train_loader, val_loader)
    
    # Get OOF predictions
    preds = trainer.predict(model, val_loader)
    y_pred = torch.cat(preds).numpy().flatten()
    chemprop_oof_preds[val_idx] = y_pred
    
    y_val = y_train[val_idx]
    rae = calculate_rae(y_val, y_pred)
    mae = mean_absolute_error(y_val, y_pred)
    r2 = r2_score(y_val, y_pred)
    
    chemprop_fold_results.append({'rae': rae, 'mae': mae, 'r2': r2})
    print(f"  RAE={rae:.4f}, MAE={mae:.4f}, R²={r2:.4f}")

chemprop_cv_rae = np.mean([r['rae'] for r in chemprop_fold_results])
chemprop_cv_mae = np.mean([r['mae'] for r in chemprop_fold_results])
chemprop_cv_r2 = np.mean([r['r2'] for r in chemprop_fold_results])

print("\n" + "="*80)
print(f"Chemprop CV Results:")
print(f"  RAE: {chemprop_cv_rae:.4f} ± {np.std([r['rae'] for r in chemprop_fold_results]):.4f}")
print(f"  MAE: {chemprop_cv_mae:.4f} ± {np.std([r['mae'] for r in chemprop_fold_results]):.4f}")
print(f"  R²:  {chemprop_cv_r2:.4f}")
print("="*80)

## 5. Install and Setup TabPFN

In [ ]:
# Check if tabpfn is available
try:
    from tabpfn import TabPFNRegressor
    print("✅ TabPFN available")
    TABPFN_AVAILABLE = True
except ImportError:
    print("❌ TabPFN not installed. Installing...")
    !pip install tabpfn -q
    try:
        from tabpfn import TabPFNRegressor
        print("✅ TabPFN installed")
        TABPFN_AVAILABLE = True
    except ImportError:
        print("⚠️  TabPFN installation failed. Will use LGBM instead.")
        TABPFN_AVAILABLE = False

## 6. Generate Features for TabPFN

TabPFN works best with classical features (not raw SMILES). We'll use:
- Morgan fingerprints (2048 bits)
- RDKit descriptors (~200 features)

In [ ]:
import useful_rdkit_utils as uru
from sklearn.impute import SimpleImputer

print("Generating features for TabPFN...")

# RDKit descriptors
print("1/2: RDKit descriptors...")
rdkit_desc = uru.RDKitDescriptors()
train_passed["rdkit_desc"] = [rdkit_desc.calc_smiles(x) for x in tqdm(train_passed.SMILES)]
test_df["rdkit_desc"] = [rdkit_desc.calc_smiles(x) for x in tqdm(test_df.SMILES)]

# Morgan fingerprints
print("\n2/2: Morgan fingerprints...")
def get_morgan_fp(smiles, radius=2, nbits=2048):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.zeros(nbits)
    return np.array(AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=nbits))

train_passed["morgan_fp"] = [get_morgan_fp(x) for x in tqdm(train_passed.SMILES)]
test_df["morgan_fp"] = [get_morgan_fp(x) for x in tqdm(test_df.SMILES)]

# Prepare feature matrices
rdkit_train = np.stack(train_passed.rdkit_desc.values)
rdkit_test = np.stack(test_df.rdkit_desc.values)

imputer = SimpleImputer(strategy='median')
rdkit_train = imputer.fit_transform(rdkit_train)
rdkit_test = imputer.transform(rdkit_test)

morgan_train = np.stack(train_passed.morgan_fp.values)
morgan_test = np.stack(test_df.morgan_fp.values)

# Normalize
scaler_rdkit = StandardScaler()
scaler_morgan = StandardScaler()

rdkit_train_norm = scaler_rdkit.fit_transform(rdkit_train)
rdkit_test_norm = scaler_rdkit.transform(rdkit_test)

morgan_train_norm = scaler_morgan.fit_transform(morgan_train)
morgan_test_norm = scaler_morgan.transform(morgan_test)

# Combine features
X_train = np.hstack([rdkit_train_norm, morgan_train_norm])
X_test = np.hstack([rdkit_test_norm, morgan_test_norm])

print(f"\nFeature matrix shape: {X_train.shape}")
print(f"Test matrix shape: {X_test.shape}")

## 7. Train TabPFN (or LGBM as fallback)

In [ ]:
if TABPFN_AVAILABLE:
    print("="*80)
    print("TABPFN: 5-FOLD SCAFFOLD-GROUPED CROSS-VALIDATION")
    print("="*80)
    print("\nNote: TabPFN has dataset size limits (~1000 samples). Using subset per fold.")
    
    tabpfn_oof_preds = np.zeros(len(train_passed))
    tabpfn_fold_results = []
    
    for fold, (train_idx, val_idx) in enumerate(group_kfold.split(X_train, y_train, scaffold_groups), 1):
        print(f"\nFold {fold}/5...")
        
        X_tr, X_val = X_train[train_idx], X_train[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]
        
        # TabPFN has limits on dataset size (typically ~1000 samples)
        # Subsample if needed
        max_samples = 1000
        if len(X_tr) > max_samples:
            print(f"  Subsampling {len(X_tr)} -> {max_samples} training samples")
            subsample_idx = np.random.choice(len(X_tr), max_samples, replace=False)
            X_tr = X_tr[subsample_idx]
            y_tr = y_tr[subsample_idx]
        
        # TabPFN also has feature limit (~100 features)
        # Use PCA or feature selection if needed
        if X_tr.shape[1] > 100:
            print(f"  Using first 100 features (TabPFN limit)")
            X_tr_subset = X_tr[:, :100]
            X_val_subset = X_val[:, :100]
        else:
            X_tr_subset = X_tr
            X_val_subset = X_val
        
        try:
            model = TabPFNRegressor(device='cpu', N_ensemble_configurations=4)
            model.fit(X_tr_subset, y_tr)
            y_pred = model.predict(X_val_subset)
            tabpfn_oof_preds[val_idx] = y_pred
            
            rae = calculate_rae(y_val, y_pred)
            mae = mean_absolute_error(y_val, y_pred)
            r2 = r2_score(y_val, y_pred)
            
            tabpfn_fold_results.append({'rae': rae, 'mae': mae, 'r2': r2})
            print(f"  RAE={rae:.4f}, MAE={mae:.4f}, R²={r2:.4f}")
        except Exception as e:
            print(f"  ⚠️  TabPFN failed: {e}")
            print(f"  Falling back to LGBM for this fold")
            TABPFN_AVAILABLE = False
            break
    
    if TABPFN_AVAILABLE and len(tabpfn_fold_results) == 5:
        tabpfn_cv_rae = np.mean([r['rae'] for r in tabpfn_fold_results])
        tabpfn_cv_mae = np.mean([r['mae'] for r in tabpfn_fold_results])
        tabpfn_cv_r2 = np.mean([r['r2'] for r in tabpfn_fold_results])
        
        print("\n" + "="*80)
        print(f"TabPFN CV Results:")
        print(f"  RAE: {tabpfn_cv_rae:.4f} ± {np.std([r['rae'] for r in tabpfn_fold_results]):.4f}")
        print(f"  MAE: {tabpfn_cv_mae:.4f} ± {np.std([r['mae'] for r in tabpfn_fold_results]):.4f}")
        print(f"  R²:  {tabpfn_cv_r2:.4f}")
        print("="*80)

if not TABPFN_AVAILABLE:
    print("\n" + "="*80)
    print("LGBM: 5-FOLD SCAFFOLD-GROUPED CROSS-VALIDATION (TabPFN fallback)")
    print("="*80)
    
    from lightgbm import LGBMRegressor
    
    lgbm_params = {
        'n_estimators': 3000,
        'learning_rate': 0.05,
        'num_leaves': 31,
        'max_depth': 7,
        'min_child_samples': 20,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'verbose': -1
    }
    
    tabpfn_oof_preds = np.zeros(len(train_passed))  # Will be LGBM preds
    tabpfn_fold_results = []
    
    for fold, (train_idx, val_idx) in enumerate(group_kfold.split(X_train, y_train, scaffold_groups), 1):
        X_tr, X_val = X_train[train_idx], X_train[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]
        
        model = LGBMRegressor(**lgbm_params, random_state=fold)
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_val)
        tabpfn_oof_preds[val_idx] = y_pred
        
        rae = calculate_rae(y_val, y_pred)
        mae = mean_absolute_error(y_val, y_pred)
        r2 = r2_score(y_val, y_pred)
        
        tabpfn_fold_results.append({'rae': rae, 'mae': mae, 'r2': r2})
        print(f"Fold {fold}: RAE={rae:.4f}, MAE={mae:.4f}, R²={r2:.4f}")
    
    tabpfn_cv_rae = np.mean([r['rae'] for r in tabpfn_fold_results])
    tabpfn_cv_mae = np.mean([r['mae'] for r in tabpfn_fold_results])
    tabpfn_cv_r2 = np.mean([r['r2'] for r in tabpfn_fold_results])
    
    print("\n" + "="*80)
    print(f"LGBM CV Results:")
    print(f"  RAE: {tabpfn_cv_rae:.4f} ± {np.std([r['rae'] for r in tabpfn_fold_results]):.4f}")
    print(f"  MAE: {tabpfn_cv_mae:.4f} ± {np.std([r['mae'] for r in tabpfn_fold_results]):.4f}")
    print(f"  R²:  {tabpfn_cv_r2:.4f}")
    print("="*80)

## 8. Ensemble: Chemprop + TabPFN/LGBM

In [ ]:
print("="*80)
print("ENSEMBLE: Chemprop + TabPFN/LGBM")
print("="*80)

# Try different ensemble weights
weights_to_try = [
    (0.5, 0.5),
    (0.6, 0.4),
    (0.4, 0.6),
    (0.7, 0.3),
    (0.3, 0.7),
]

best_ensemble_rae = float('inf')
best_weights = None

print("\nTesting ensemble weights (Chemprop, TabPFN/LGBM):")
for w1, w2 in weights_to_try:
    ensemble_preds = w1 * chemprop_oof_preds + w2 * tabpfn_oof_preds
    
    rae = calculate_rae(y_train, ensemble_preds)
    mae = mean_absolute_error(y_train, ensemble_preds)
    r2 = r2_score(y_train, ensemble_preds)
    
    print(f"  ({w1:.1f}, {w2:.1f}): RAE={rae:.4f}, MAE={mae:.4f}, R²={r2:.4f}")
    
    if rae < best_ensemble_rae:
        best_ensemble_rae = rae
        best_weights = (w1, w2)

print(f"\n✅ Best weights: Chemprop={best_weights[0]:.1f}, TabPFN/LGBM={best_weights[1]:.1f}")
print(f"✅ Best ensemble RAE: {best_ensemble_rae:.4f}")

# Compare all approaches
print("\n" + "="*80)
print("COMPARISON: All Approaches")
print("="*80)
comparison_df = pd.DataFrame([
    {'Method': 'Chemprop', 'CV RAE': chemprop_cv_rae, 'CV MAE': chemprop_cv_mae},
    {'Method': 'TabPFN/LGBM', 'CV RAE': tabpfn_cv_rae, 'CV MAE': tabpfn_cv_mae},
    {'Method': f'Ensemble ({best_weights[0]:.1f}, {best_weights[1]:.1f})', 'CV RAE': best_ensemble_rae, 
     'CV MAE': mean_absolute_error(y_train, best_weights[0]*chemprop_oof_preds + best_weights[1]*tabpfn_oof_preds)},
])
print(comparison_df.to_string(index=False))
print("="*80)

## 9. Train Final Models and Generate Predictions

In [ ]:
print("Training final models on full training set...")
print("\n1/2: Chemprop...")

# Chemprop final model
train_smiles_all = train_passed['SMILES'].tolist()
train_targets_all = y_train.reshape(-1, 1).tolist()
test_smiles_all = test_df['SMILES'].tolist()

featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer()

train_data_full = cpdata.MoleculeDataset(
    [cpdata.MoleculeDatapoint.from_smi(smi, y)
     for smi, y in zip(train_smiles_all, train_targets_all)],
    featurizer=featurizer
)
test_data_full = cpdata.MoleculeDataset(
    [cpdata.MoleculeDatapoint.from_smi(smi) for smi in test_smiles_all],
    featurizer=featurizer
)

train_loader_full = cpdata.build_dataloader(train_data_full, shuffle=True, batch_size=50)
test_loader_full = cpdata.build_dataloader(test_data_full, shuffle=False, batch_size=50)

mp = cpnn.BondMessagePassing()
agg = cpnn.MeanAggregation()
ffn = cpnn.RegressionFFN()
chemprop_final = cpmodels.MPNN(mp, agg, ffn)

trainer = pl.Trainer(
    max_epochs=50,
    enable_progress_bar=True,
    enable_model_summary=False,
)
trainer.fit(chemprop_final, train_loader_full)

chemprop_test_preds = trainer.predict(chemprop_final, test_loader_full)
chemprop_test_preds = torch.cat(chemprop_test_preds).numpy().flatten()

print(f"\nChemprop test predictions: mean={chemprop_test_preds.mean():.3f}, std={chemprop_test_preds.std():.3f}")

In [ ]:
print("\n2/2: TabPFN/LGBM...")

if TABPFN_AVAILABLE:
    # TabPFN with subsampling and feature selection
    max_samples = 1000
    if len(X_train) > max_samples:
        subsample_idx = np.random.choice(len(X_train), max_samples, replace=False)
        X_train_subset = X_train[subsample_idx, :100]
        y_train_subset = y_train[subsample_idx]
    else:
        X_train_subset = X_train[:, :100]
        y_train_subset = y_train
    
    tabpfn_final = TabPFNRegressor(device='cpu', N_ensemble_configurations=4)
    tabpfn_final.fit(X_train_subset, y_train_subset)
    tabpfn_test_preds = tabpfn_final.predict(X_test[:, :100])
else:
    # LGBM
    lgbm_final = LGBMRegressor(**lgbm_params, random_state=42)
    lgbm_final.fit(X_train, y_train)
    tabpfn_test_preds = lgbm_final.predict(X_test)

print(f"TabPFN/LGBM test predictions: mean={tabpfn_test_preds.mean():.3f}, std={tabpfn_test_preds.std():.3f}")

# Ensemble predictions
ensemble_test_preds = best_weights[0] * chemprop_test_preds + best_weights[1] * tabpfn_test_preds

print(f"\n✅ Ensemble test predictions: mean={ensemble_test_preds.mean():.3f}, std={ensemble_test_preds.std():.3f}")
print(f"Training set mean: {y_train.mean():.3f}")

## 10. Visualize Predictions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Chemprop
axes[0].hist(chemprop_test_preds, bins=30, alpha=0.7, color='steelblue', edgecolor='black')
axes[0].axvline(y_train.mean(), color='red', linestyle='--', linewidth=2, label='Train mean')
axes[0].set_xlabel('pEC50')
axes[0].set_ylabel('Count')
axes[0].set_title('Chemprop Predictions')
axes[0].legend()

# TabPFN/LGBM
axes[1].hist(tabpfn_test_preds, bins=30, alpha=0.7, color='green', edgecolor='black')
axes[1].axvline(y_train.mean(), color='red', linestyle='--', linewidth=2, label='Train mean')
axes[1].set_xlabel('pEC50')
axes[1].set_ylabel('Count')
axes[1].set_title('TabPFN/LGBM Predictions')
axes[1].legend()

# Ensemble
axes[2].hist(ensemble_test_preds, bins=30, alpha=0.7, color='coral', edgecolor='black')
axes[2].axvline(y_train.mean(), color='red', linestyle='--', linewidth=2, label='Train mean')
axes[2].set_xlabel('pEC50')
axes[2].set_ylabel('Count')
axes[2].set_title(f'Ensemble ({best_weights[0]:.1f}, {best_weights[1]:.1f})')
axes[2].legend()

plt.tight_layout()
plt.show()

## 11. Create Submission

In [ ]:
submission_df = pd.DataFrame({
    'SMILES': test_df['SMILES'],
    'Molecule Name': test_df['Molecule Name'],
    'pEC50': ensemble_test_preds
})

# Save submission
output_path = '../outputs/day10_chemprop_tabpfn_ensemble_submission.csv'
submission_df.to_csv(output_path, index=False)

print(f"✅ Submission saved to: {output_path}")
print(f"Shape: {submission_df.shape}")
print("\nFirst 10 predictions:")
print(submission_df.head(10))

## 12. Validate Submission

In [ ]:
import sys
from pathlib import Path

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from validation.activity_validation import validate_activity_submission

expected_activity_ids = set(test_df["Molecule Name"])
is_valid, validation_errors = validate_activity_submission(
    Path(output_path),
    expected_ids=expected_activity_ids,
)

if is_valid:
    print("✅ Submission file is valid!")
else:
    print("❌ Submission file is invalid:")
    for msg in validation_errors:
        print(f" - {msg}")

## Summary

**Key findings:**
1. Trained on ~2800 molecules (counter-screen passed only)
2. Chemprop (message passing NN) CV performance: ?
3. TabPFN/LGBM CV performance: ?
4. Ensemble CV performance: ?

**Comparison to baseline (day 9):**
- Day 9: LGBM + RDKit + Morgan on 4139 molecules → 0.577 RAE, 0.52 MAE
- Day 10: Chemprop + TabPFN on 2800 molecules → ? RAE, ? MAE

**Discord target: 0.44 MAE**

**Next steps if not reaching 0.44 MAE:**
1. Hyperparameter tuning for Chemprop (epochs, learning rate, hidden dims)
2. Try more sophisticated ensembling (stacking with Ridge meta-learner)
3. Feature engineering for TabPFN
4. Data augmentation (SMILES enumeration)
5. Check if test set assumption is wrong (maybe some failed counter-screen?)